# XAI Insertion/Deletion: Impact of Pixel Replacement Strategy

**Master's Thesis Experiment — Data Science**

**Research question:** How does the choice of pixel replacement strategy affect the Insertion/Deletion faithfulness metrics for CNN explainability methods?

**Replacement strategies compared:**
1. Zero/black pixels (standard baseline)
2. Gaussian-blurred original image
3. Mean background (SHAP-style, training-set mean)
4. Random Gaussian noise
5. Generative inpainting (OpenCV Telea; optionally Stable Diffusion)

**XAI methods evaluated:** GradCAM, Integrated Gradients, LIME, SHAP

**Model:** ResNet50 pretrained on ImageNet  
**Dataset:** ImageNette2-160 (10-class ImageNet subset)

In [ ]:
# ── Section 0: Install dependencies ───────────────────────────────────────────
!pip install captum lime shap diffusers accelerate timm -q
# Verify GPU
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected')

In [ ]:
# ── Section 1: Imports and Configuration ──────────────────────────────────────
import os, gc, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from scipy.stats import kendalltau
import cv2

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision import models

# captum
from captum.attr import LayerGradCam, IntegratedGradients
from captum.attr import visualization as viz

# lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

# shap
import shap

warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── ImageNet normalization constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

# ── ImageNette class mapping (WordNet ID → human label)
IMAGENETTE_WNIDS = {
    'n01440764': 'tench',
    'n02102040': 'English springer',
    'n02979186': 'cassette player',
    'n03000684': 'chain saw',
    'n03028079': 'church',
    'n03394916': 'French horn',
    'n03417042': 'garbage truck',
    'n03425413': 'gas pump',
    'n03445777': 'golf ball',
    'n03888257': 'parachute'
}

# ImageNet class indices for each ImageNette class
IMAGENETTE_IMAGENET_IDX = {
    'n01440764': 0,   'n02102040': 217, 'n02979186': 482,
    'n03000684': 491, 'n03028079': 497, 'n03394916': 566,
    'n03417042': 569, 'n03425413': 571, 'n03445777': 574,
    'n03888257': 701
}

# matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
FIGSIZE = (12, 8)
DPI = 150

print('All imports successful.')

In [ ]:
# ── Section 2: Load Dataset (ImageNette2-160) ──────────────────────────────────
DATA_DIR = Path('data/imagenette2-160')

if not DATA_DIR.exists():
    print('Downloading ImageNette2-160...')
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -O data/imagenette2-160.tgz
    !mkdir -p data && tar -xzf data/imagenette2-160.tgz -C data/
    print('Done.')
else:
    print('Dataset already downloaded.')

# ── Transforms
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

transform_no_norm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor()
])

# ── Load validation set, 20 images per class = 200 total
val_dir = DATA_DIR / 'val'
images_data = []  # list of (img_tensor, img_pil, label_imagenet_idx, class_name, img_path)

PER_CLASS = 20
for wnid, label_name in IMAGENETTE_WNIDS.items():
    class_dir = val_dir / wnid
    if not class_dir.exists():
        continue
    img_paths = sorted(class_dir.glob('*.JPEG'))[:PER_CLASS]
    for p in img_paths:
        pil = Image.open(p).convert('RGB')
        tensor = transform(pil)
        imagenet_idx = IMAGENETTE_IMAGENET_IDX[wnid]
        images_data.append((tensor, pil, imagenet_idx, label_name, str(p)))

print(f'Loaded {len(images_data)} images across {len(IMAGENETTE_WNIDS)} classes')

# ── Build background dataset (20 images from train set, 2 per class)
train_dir = DATA_DIR / 'train'
bg_tensors = []
for wnid in IMAGENETTE_WNIDS:
    class_dir = train_dir / wnid
    if not class_dir.exists():
        continue
    for p in sorted(class_dir.glob('*.JPEG'))[:2]:
        pil = Image.open(p).convert('RGB')
        bg_tensors.append(transform(pil))

BG_TENSOR = torch.stack(bg_tensors)  # (20, 3, 224, 224)
BG_MEAN = BG_TENSOR.mean(dim=0)      # (3, 224, 224)
print(f'Background dataset: {BG_TENSOR.shape}, mean shape: {BG_MEAN.shape}')

# ── Visualize sample grid
fig, axes = plt.subplots(2, 5, figsize=(14, 6), dpi=DPI)
unnorm = T.Compose([
    T.Normalize(mean=[0,0,0], std=[1/s for s in IMAGENET_STD]),
    T.Normalize(mean=[-m for m in IMAGENET_MEAN], std=[1,1,1])
])
shown_classes = set()
idx_plot = 0
for tensor, pil, imgnet_idx, name, _ in images_data:
    if name in shown_classes:
        continue
    shown_classes.add(name)
    ax = axes.flat[idx_plot]
    ax.imshow(pil.resize((IMG_SIZE, IMG_SIZE)))
    ax.set_title(name, fontsize=9)
    ax.axis('off')
    idx_plot += 1
    if idx_plot >= 10:
        break
plt.suptitle('ImageNette2-160 — One sample per class', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_sample_grid.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 3: Load Pre-trained Model ─────────────────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model = model.to(DEVICE)
model.eval()
print('ResNet50 loaded. Parameters:', sum(p.numel() for p in model.parameters()) // 1_000_000, 'M')

# Load ImageNet class labels
import urllib.request
labels_url = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
with urllib.request.urlopen(labels_url) as f:
    IMAGENET_LABELS = json.load(f)

def predict_topk(model, img_tensor, k=5):
    """Return top-k (class_idx, label, confidence) tuples."""
    with torch.no_grad():
        inp = img_tensor.unsqueeze(0).to(DEVICE)
        logits = model(inp)
        probs = F.softmax(logits, dim=1)[0]
    topk_probs, topk_idxs = probs.topk(k)
    return [(int(i), IMAGENET_LABELS[int(i)], float(p))
            for i, p in zip(topk_idxs, topk_probs)]

# Show top-5 for a sample
sample_tensor, sample_pil, sample_cls, sample_name, _ = images_data[0]
print(f'Sample class: {sample_name}')
for idx, label, conf in predict_topk(model, sample_tensor):
    print(f'  [{idx:4d}] {label:<35s} {conf:.4f}')

fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=DPI)
ax.imshow(sample_pil.resize((IMG_SIZE, IMG_SIZE)))
ax.set_title(f'Ground truth: {sample_name}', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Section 4: XAI Methods ────────────────────────────────────────────────────
# All methods return a 2D numpy array of shape (H, W) normalised to [0,1]

def _normalize_saliency(sal):
    """Normalise a 2D map to [0, 1]."""
    sal = np.abs(sal)
    sal_min, sal_max = sal.min(), sal.max()
    if sal_max - sal_min < 1e-8:
        return np.zeros_like(sal)
    return (sal - sal_min) / (sal_max - sal_min)


# ── 4a GradCAM ────────────────────────────────────────────────────────────────
def compute_gradcam(model, img_tensor, target_class):
    """LayerGradCam on ResNet50's layer4."""
    model.zero_grad()
    lgc = LayerGradCam(model, model.layer4)
    inp = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
    attr = lgc.attribute(inp, target=target_class)
    # attr shape: (1, C, h, w) → upsample to IMG_SIZE
    attr_up = F.interpolate(
        attr.sum(dim=1, keepdim=True),
        size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False
    )
    sal = attr_up.squeeze().detach().cpu().numpy()
    return _normalize_saliency(sal)


# ── 4b Integrated Gradients ───────────────────────────────────────────────────
def compute_integrated_gradients(model, img_tensor, target_class, n_steps=50):
    """IntegratedGradients from captum, baseline = zero tensor."""
    ig = IntegratedGradients(model)
    inp = img_tensor.unsqueeze(0).to(DEVICE)
    baseline = torch.zeros_like(inp)
    attr = ig.attribute(inp, baselines=baseline, target=target_class,
                        n_steps=n_steps, internal_batch_size=10)
    sal = attr.squeeze().detach().cpu().numpy()  # (3, H, W)
    sal = sal.sum(axis=0)  # sum over channels → (H, W)
    return _normalize_saliency(sal)


# ── 4c LIME ───────────────────────────────────────────────────────────────────
def compute_lime(model, img_pil, target_class, num_samples=500):
    """LIME with superpixel segmentation. Returns 2D saliency map."""
    explainer = lime_image.LimeImageExplainer()

    def predict_fn(imgs_np):  # (N, H, W, 3) uint8
        batch = []
        for img_np in imgs_np:
            pil = Image.fromarray(img_np.astype(np.uint8))
            t = transform(pil)
            batch.append(t)
        batch = torch.stack(batch).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(batch), dim=1).cpu().numpy()
        return probs

    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
    explanation = explainer.explain_instance(
        img_np, predict_fn,
        top_labels=5,
        hide_color=0,
        num_samples=num_samples,
        random_seed=SEED
    )
    # Get weight per superpixel for target_class
    if target_class not in explanation.local_exp:
        # fall back to top predicted
        target_class = explanation.top_labels[0]
    segments = explanation.segments  # (H, W) int
    local_exp = dict(explanation.local_exp[target_class])
    sal = np.zeros(segments.shape, dtype=np.float32)
    for seg_id, weight in local_exp.items():
        sal[segments == seg_id] = weight
    return _normalize_saliency(sal)


# ── 4d SHAP ───────────────────────────────────────────────────────────────────
def compute_shap(model, img_tensor, background_tensors, target_class):
    """SHAP GradientExplainer. background_tensors shape: (N,3,H,W)."""
    bg = background_tensors.to(DEVICE)
    e = shap.GradientExplainer(model, bg)
    inp = img_tensor.unsqueeze(0).to(DEVICE)
    shap_vals = e.shap_values(inp, ranked_outputs=1, nsamples=200)
    # shap_vals is list of arrays, shape (1,3,H,W)
    if isinstance(shap_vals, list):
        vals = shap_vals[0]  # for top-1 output
    else:
        vals = shap_vals
    sal = np.abs(vals).sum(axis=1)[0]  # sum over channels → (H, W)
    return _normalize_saliency(sal)


# ── Dispatcher ────────────────────────────────────────────────────────────────
def compute_saliency(method, model, img_tensor, img_pil, target_class, bg_tensors=None):
    if method == 'gradcam':
        return compute_gradcam(model, img_tensor, target_class)
    elif method == 'ig':
        return compute_integrated_gradients(model, img_tensor, target_class)
    elif method == 'lime':
        return compute_lime(model, img_pil, target_class)
    elif method == 'shap':
        return compute_shap(model, img_tensor, bg_tensors, target_class)
    else:
        raise ValueError(f'Unknown method: {method}')


# ── Visualise saliency maps for one sample image
def overlay_saliency(pil_img, sal_map, alpha=0.5, cmap='jet'):
    img_arr = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    heatmap = plt.cm.get_cmap(cmap)(sal_map)[:, :, :3]
    return (1 - alpha) * img_arr + alpha * heatmap

print('Computing saliency maps for a sample image — this may take ~1 minute...')
s_tensor, s_pil, s_cls, s_name, _ = images_data[2]

sals = {
    'GradCAM': compute_gradcam(model, s_tensor, s_cls),
    'Integrated\nGradients': compute_integrated_gradients(model, s_tensor, s_cls),
    'LIME': compute_lime(model, s_pil, s_cls),
    'SHAP': compute_shap(model, s_tensor, BG_TENSOR[:20], s_cls),
}

fig, axes = plt.subplots(1, 5, figsize=(16, 4), dpi=DPI)
axes[0].imshow(s_pil.resize((IMG_SIZE, IMG_SIZE)))
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')
for ax, (name, sal) in zip(axes[1:], sals.items()):
    ax.imshow(overlay_saliency(s_pil, sal))
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.suptitle(f'Saliency maps — class: {s_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_saliency_maps.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 5: Replacement Strategies ─────────────────────────────────────────
# mask: numpy array (H, W), values in {0,1} — 1=keep original, 0=replace

def tensor_to_numpy_rgb(img_tensor):
    """Unnormalise tensor (3,H,W) → uint8 numpy (H,W,3)."""
    t = img_tensor.clone().cpu()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        t[c] = t[c] * s + m
    t = t.permute(1, 2, 0).clamp(0, 1).numpy()
    return (t * 255).astype(np.uint8)

def numpy_rgb_to_tensor(img_np):
    """uint8 numpy (H,W,3) → normalised tensor (3,H,W)."""
    t = torch.from_numpy(img_np.astype(np.float32) / 255.0).permute(2, 0, 1)
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        t[c] = (t[c] - m) / s
    return t


def replace_black(img_tensor, mask):
    """Replace masked pixels with normalised black (zero in orig space)."""
    result = img_tensor.clone()
    # 'black' in normalised space = -mean/std per channel
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        result[c][mask == 0] = -m / s
    return result


def replace_blur(img_tensor, mask, sigma=10):
    """Replace masked pixels with values from a Gaussian-blurred version."""
    img_np = tensor_to_numpy_rgb(img_tensor)  # (H,W,3) uint8
    ksize = int(6 * sigma + 1) | 1  # odd kernel
    blurred = cv2.GaussianBlur(img_np, (ksize, ksize), sigma)
    result_np = img_np.copy()
    result_np[mask == 0] = blurred[mask == 0]
    return numpy_rgb_to_tensor(result_np)


def replace_mean_background(img_tensor, mask, bg_mean_tensor):
    """Replace masked pixels with training-set mean (SHAP-style)."""
    result = img_tensor.clone()
    for c in range(3):
        result[c][mask == 0] = bg_mean_tensor[c][mask == 0]
    return result


def replace_noise(img_tensor, mask, std=0.1):
    """Replace masked pixels with Gaussian noise (in normalised space)."""
    result = img_tensor.clone()
    noise = torch.randn_like(img_tensor) * std
    for c in range(3):
        result[c][mask == 0] = noise[c][mask == 0]
    return result


def replace_inpaint_cv2(img_tensor, mask):
    """OpenCV Telea inpainting. mask=0 → inpaint."""
    img_np = tensor_to_numpy_rgb(img_tensor)
    # OpenCV inpaint expects uint8 mask where 255 = inpaint
    cv_mask = ((1 - mask) * 255).astype(np.uint8)
    # Dilate mask slightly so edges are covered
    kernel = np.ones((3, 3), np.uint8)
    cv_mask_dilated = cv2.dilate(cv_mask, kernel, iterations=1)
    inpainted = cv2.inpaint(img_np, cv_mask_dilated, inpaintRadius=3,
                             flags=cv2.INPAINT_TELEA)
    result_np = img_np.copy()
    result_np[mask == 0] = inpainted[mask == 0]
    return numpy_rgb_to_tensor(result_np)


def replace_inpaint_sd(img_tensor, mask, pipe):
    """Stable Diffusion inpainting (requires diffusers pipe loaded separately)."""
    try:
        img_np = tensor_to_numpy_rgb(img_tensor)
        pil_img = Image.fromarray(img_np)
        # mask: 1=keep, 0=replace → SD expects white=inpaint, black=keep
        mask_pil = Image.fromarray(((1 - mask) * 255).astype(np.uint8), mode='L')
        result_pil = pipe(
            prompt='a natural scene',
            image=pil_img,
            mask_image=mask_pil,
            height=IMG_SIZE,
            width=IMG_SIZE,
            num_inference_steps=20,
            guidance_scale=7.5
        ).images[0]
        result_np = np.array(result_pil)
        # Blend: only replace masked pixels
        composite = img_np.copy()
        composite[mask == 0] = result_np[mask == 0]
        return numpy_rgb_to_tensor(composite)
    except Exception:
        # Fallback to blur if SD fails
        return replace_blur(img_tensor, mask)


def get_replace_fn(strategy, bg_mean_tensor=None, sd_pipe=None):
    """Return a replacement function (img_tensor, mask) -> tensor."""
    if strategy == 'black':
        return lambda t, m: replace_black(t, m)
    elif strategy == 'blur':
        return lambda t, m: replace_blur(t, m)
    elif strategy == 'mean_bg':
        return lambda t, m: replace_mean_background(t, m, bg_mean_tensor)
    elif strategy == 'noise':
        return lambda t, m: replace_noise(t, m)
    elif strategy == 'inpaint_cv2':
        return lambda t, m: replace_inpaint_cv2(t, m)
    elif strategy == 'inpaint_sd':
        return lambda t, m: replace_inpaint_sd(t, m, sd_pipe)
    else:
        raise ValueError(f'Unknown strategy: {strategy}')


# ── Visual comparison of strategies at 50% mask
demo_tensor, demo_pil, demo_cls, demo_name, _ = images_data[1]

# Random 50% mask
rng = np.random.default_rng(SEED)
demo_mask_flat = np.ones(IMG_SIZE * IMG_SIZE)
demo_mask_flat[:IMG_SIZE * IMG_SIZE // 2] = 0
rng.shuffle(demo_mask_flat)
demo_mask = demo_mask_flat.reshape(IMG_SIZE, IMG_SIZE)

strategies = ['black', 'blur', 'mean_bg', 'noise', 'inpaint_cv2']
strategy_labels = ['Black', 'Blur', 'Mean BG', 'Noise', 'CV2 Inpaint']

fig, axes = plt.subplots(1, 6, figsize=(18, 4), dpi=DPI)
axes[0].imshow(tensor_to_numpy_rgb(demo_tensor))
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')
for ax, strat, label in zip(axes[1:], strategies, strategy_labels):
    rfn = get_replace_fn(strat, bg_mean_tensor=BG_MEAN)
    replaced = rfn(demo_tensor, demo_mask)
    ax.imshow(tensor_to_numpy_rgb(replaced))
    ax.set_title(label, fontsize=10)
    ax.axis('off')
plt.suptitle('Replacement strategies — 50% pixels masked', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_replacement_strategies.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 6: Insertion and Deletion Metrics ─────────────────────────────────

def _get_confidence(model, img_tensor, target_class):
    """Scalar confidence for target_class."""
    with torch.no_grad():
        inp = img_tensor.unsqueeze(0).to(DEVICE)
        probs = F.softmax(model(inp), dim=1)[0]
    return float(probs[target_class])


def compute_deletion_curve(model, img_tensor, saliency_map, replace_fn,
                            target_class, n_steps=50):
    """
    Deletion: start from original, progressively delete (replace) pixels
    in order of decreasing saliency.
    Returns array of confidence scores at each step.
    """
    H, W = saliency_map.shape
    # Rank pixels: most salient first
    flat_sal = saliency_map.flatten()
    sorted_idx = np.argsort(flat_sal)[::-1]  # descending
    total_pixels = H * W
    step_size = total_pixels // n_steps

    scores = []
    mask = np.ones(total_pixels, dtype=np.float32)  # all kept

    for step in range(n_steps + 1):
        cur_mask = mask.reshape(H, W)
        replaced = replace_fn(img_tensor, cur_mask)
        scores.append(_get_confidence(model, replaced, target_class))
        # Delete next chunk
        start = step * step_size
        end = min((step + 1) * step_size, total_pixels)
        mask[sorted_idx[start:end]] = 0

    return np.array(scores)


def compute_insertion_curve(model, img_tensor, saliency_map, replace_fn,
                             target_class, n_steps=50):
    """
    Insertion: start from fully replaced image, progressively reveal pixels
    in order of decreasing saliency.
    Returns array of confidence scores at each step.
    """
    H, W = saliency_map.shape
    flat_sal = saliency_map.flatten()
    sorted_idx = np.argsort(flat_sal)[::-1]  # descending
    total_pixels = H * W
    step_size = total_pixels // n_steps

    scores = []
    mask = np.zeros(total_pixels, dtype=np.float32)  # all replaced

    for step in range(n_steps + 1):
        cur_mask = mask.reshape(H, W)
        replaced = replace_fn(img_tensor, cur_mask)
        scores.append(_get_confidence(model, replaced, target_class))
        # Insert next chunk
        start = step * step_size
        end = min((step + 1) * step_size, total_pixels)
        mask[sorted_idx[start:end]] = 1

    return np.array(scores)


def auc_score(curve):
    """Normalised area under curve using trapezoidal rule."""
    return float(np.trapz(curve) / (len(curve) - 1))


# ── Demo: curves for one image, one XAI method, all strategies
print('Generating demo curves (this may take 2-3 minutes)...')
demo_sal = compute_gradcam(model, demo_tensor, demo_cls)

demo_ins_curves = {}
demo_del_curves = {}
for strat, label in zip(strategies, strategy_labels):
    rfn = get_replace_fn(strat, bg_mean_tensor=BG_MEAN)
    demo_ins_curves[label] = compute_insertion_curve(
        model, demo_tensor, demo_sal, rfn, demo_cls, n_steps=20)
    demo_del_curves[label] = compute_deletion_curve(
        model, demo_tensor, demo_sal, rfn, demo_cls, n_steps=20)
    print(f'  {label}: INS AUC={auc_score(demo_ins_curves[label]):.3f} | DEL AUC={auc_score(demo_del_curves[label]):.3f}')

x_axis = np.linspace(0, 1, 21)
colors = plt.cm.tab10(np.linspace(0, 1, len(strategies)))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, dpi=DPI)
for (label, curve), color in zip(demo_ins_curves.items(), colors):
    axes[0].plot(x_axis, curve, label=label, color=color, lw=2)
axes[0].set_title('Insertion Curves (GradCAM)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Fraction of pixels revealed')
axes[0].set_ylabel('Model confidence')
axes[0].legend(fontsize=9)

for (label, curve), color in zip(demo_del_curves.items(), colors):
    axes[1].plot(x_axis, curve, label=label, color=color, lw=2)
axes[1].set_title('Deletion Curves (GradCAM)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Fraction of pixels deleted')
axes[1].set_ylabel('Model confidence')
axes[1].legend(fontsize=9)

plt.suptitle(f'Demo: {demo_name} — impact of replacement strategy', fontsize=12)
plt.tight_layout()
plt.savefig('fig_demo_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 7: Full Experiment ─────────────────────────────────────────────────
# 50 images × 4 XAI methods × 5 strategies = 1000 (ins + del) AUC scores
# Estimated runtime on T4: ~25-40 minutes

N_IMAGES = 50  # 5 per class
XAI_METHODS = ['gradcam', 'ig', 'lime', 'shap']
STRATEGIES  = ['black', 'blur', 'mean_bg', 'noise', 'inpaint_cv2']
N_STEPS     = 20   # coarser for speed; use 50 for publication quality

# Select 5 images per class
experiment_images = []
counts = {name: 0 for name in IMAGENETTE_WNIDS.values()}
for item in images_data:
    _, _, _, name, _ = item
    if counts[name] < 5:
        experiment_images.append(item)
        counts[name] += 1
    if len(experiment_images) == N_IMAGES:
        break
print(f'Experiment images: {len(experiment_images)}')

results = []

for img_idx, (img_tensor, img_pil, img_cls, img_name, _) in enumerate(
        tqdm(experiment_images, desc='Images')):

    for xai_method in XAI_METHODS:
        try:
            sal = compute_saliency(
                xai_method, model, img_tensor, img_pil, img_cls,
                bg_tensors=BG_TENSOR[:20]
            )
        except Exception as e:
            print(f'  Saliency error [{xai_method}, img {img_idx}]: {e}')
            continue

        for strategy in STRATEGIES:
            try:
                rfn = get_replace_fn(strategy, bg_mean_tensor=BG_MEAN)
                ins_curve = compute_insertion_curve(
                    model, img_tensor, sal, rfn, img_cls, n_steps=N_STEPS)
                del_curve = compute_deletion_curve(
                    model, img_tensor, sal, rfn, img_cls, n_steps=N_STEPS)

                results.append({
                    'img_idx':   img_idx,
                    'class':     img_name,
                    'xai':       xai_method,
                    'strategy':  strategy,
                    'ins_auc':   auc_score(ins_curve),
                    'del_auc':   auc_score(del_curve),
                    'ins_curve': ins_curve.tolist(),
                    'del_curve': del_curve.tolist(),
                })
            except Exception as e:
                print(f'  Strategy error [{strategy}]: {e}')

    # Memory cleanup
    if img_idx % 10 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df = pd.DataFrame(results)
df_save = df.drop(columns=['ins_curve', 'del_curve'])
df_save.to_csv('xai_experiment_results.csv', index=False)
print(f'Results saved. Total rows: {len(df)}')
print(df_save.head(10))

In [ ]:
# ── Section 8: Analysis and Visualizations ────────────────────────────────────
# (If re-running analysis without re-running experiment, load CSV here)
# df_save = pd.read_csv('xai_experiment_results.csv')

XAI_LABELS = {'gradcam': 'GradCAM', 'ig': 'Int. Grad.', 'lime': 'LIME', 'shap': 'SHAP'}
STRAT_LABELS = {'black': 'Black', 'blur': 'Blur', 'mean_bg': 'Mean BG',
                'noise': 'Noise', 'inpaint_cv2': 'CV2 Inpaint'}

df_plot = df_save.copy()
df_plot['xai_label'] = df_plot['xai'].map(XAI_LABELS)
df_plot['strat_label'] = df_plot['strategy'].map(STRAT_LABELS)

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 — Heatmap: mean Insertion AUC and Deletion AUC
# ─────────────────────────────────────────────────────────────────────────────
pivot_ins = df_plot.pivot_table(values='ins_auc', index='strat_label',
                                 columns='xai_label', aggfunc='mean')
pivot_del = df_plot.pivot_table(values='del_auc', index='strat_label',
                                 columns='xai_label', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=DPI)
sns.heatmap(pivot_ins, annot=True, fmt='.3f', cmap='YlOrRd',
            ax=axes[0], linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'AUC'})
axes[0].set_title('Insertion AUC\n(higher = better attribution)', fontweight='bold')
axes[0].set_xlabel('XAI Method')
axes[0].set_ylabel('Replacement Strategy')

sns.heatmap(pivot_del, annot=True, fmt='.3f', cmap='YlOrRd_r',
            ax=axes[1], linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'AUC'})
axes[1].set_title('Deletion AUC\n(lower = better attribution)', fontweight='bold')
axes[1].set_xlabel('XAI Method')
axes[1].set_ylabel('')

plt.suptitle('Plot 1 — Mean AUC: XAI method × replacement strategy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_heatmap_auc.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 — Grouped bar chart: insertion AUC per strategy × XAI method
# ─────────────────────────────────────────────────────────────────────────────
xai_order = list(XAI_LABELS.values())
strat_order = list(STRAT_LABELS.values())
colors_bar = sns.color_palette('Set2', len(strat_order))

means = df_plot.groupby(['xai_label', 'strat_label'])['ins_auc'].mean().unstack('strat_label')
stds  = df_plot.groupby(['xai_label', 'strat_label'])['ins_auc'].std().unstack('strat_label')

means = means.reindex(index=xai_order, columns=strat_order)
stds  = stds.reindex(index=xai_order, columns=strat_order)

x = np.arange(len(xai_order))
width = 0.15

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
for i, (strat, color) in enumerate(zip(strat_order, colors_bar)):
    offset = (i - len(strat_order)/2 + 0.5) * width
    ax.bar(x + offset, means[strat], width, label=strat,
           color=color, yerr=stds[strat], capsize=4, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(xai_order, fontsize=11)
ax.set_ylabel('Insertion AUC (mean ± std)', fontsize=11)
ax.set_title('Plot 2 — Insertion AUC by XAI Method and Replacement Strategy',
             fontsize=12, fontweight='bold')
ax.legend(title='Strategy', fontsize=9, title_fontsize=10)
plt.tight_layout()
plt.savefig('fig_grouped_bar.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 — Example insertion/deletion curves from stored data
# ─────────────────────────────────────────────────────────────────────────────
# Re-generate curves for one image with GradCAM (from df, pick img_idx=0)
example_img = experiment_images[0]
ex_tensor, ex_pil, ex_cls, ex_name, _ = example_img
ex_sal = compute_gradcam(model, ex_tensor, ex_cls)

ex_ins = {}
ex_del = {}
for strat, slabel in STRAT_LABELS.items():
    rfn = get_replace_fn(strat, bg_mean_tensor=BG_MEAN)
    ex_ins[slabel] = compute_insertion_curve(model, ex_tensor, ex_sal, rfn, ex_cls, n_steps=N_STEPS)
    ex_del[slabel] = compute_deletion_curve(model, ex_tensor, ex_sal, rfn, ex_cls, n_steps=N_STEPS)

x_axis = np.linspace(0, 1, N_STEPS + 1)
colors_line = sns.color_palette('tab10', len(STRAT_LABELS))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, dpi=DPI)
for (slabel, curve), color in zip(ex_ins.items(), colors_line):
    axes[0].plot(x_axis, curve, label=f'{slabel} (AUC={auc_score(curve):.2f})',
                 color=color, lw=2.0)
axes[0].fill_between(x_axis, 0, 1, alpha=0.03, color='gray')
axes[0].set_title('Insertion Curve — GradCAM', fontweight='bold')
axes[0].set_xlabel('Fraction of pixels inserted')
axes[0].set_ylabel('Model confidence')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1)

for (slabel, curve), color in zip(ex_del.items(), colors_line):
    axes[1].plot(x_axis, curve, label=f'{slabel} (AUC={auc_score(curve):.2f})',
                 color=color, lw=2.0)
axes[1].set_title('Deletion Curve — GradCAM', fontweight='bold')
axes[1].set_xlabel('Fraction of pixels deleted')
axes[1].set_ylabel('Model confidence')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1)

plt.suptitle(f'Plot 3 — Insertion/Deletion Curves: {ex_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_curves_example.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 — OoD Sensitivity: model confidence on fully-replaced images
# (How OoD is each strategy? Confidence should be ~0 for correct class)
# ─────────────────────────────────────────────────────────────────────────────
print('Computing OoD confidence for fully-replaced images...')
ood_records = []
all_ones_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)  # all replaced

for img_idx, (img_tensor, img_pil, img_cls, img_name, _) in enumerate(
        tqdm(experiment_images, desc='OoD eval')):
    for strat, slabel in STRAT_LABELS.items():
        rfn = get_replace_fn(strat, bg_mean_tensor=BG_MEAN)
        fully_replaced = rfn(img_tensor, all_ones_mask)
        conf = _get_confidence(model, fully_replaced, img_cls)
        # Also record top-1 confidence (max over all classes)
        with torch.no_grad():
            inp = fully_replaced.unsqueeze(0).to(DEVICE)
            probs = F.softmax(model(inp), dim=1)[0].cpu().numpy()
        ood_records.append({
            'strategy': strat, 'strat_label': slabel,
            'correct_cls_conf': conf,
            'max_conf': float(probs.max()),
            'entropy': float(-np.sum(probs * np.log(probs + 1e-12)))
        })

df_ood = pd.DataFrame(ood_records)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=DPI)

# Correct-class confidence
sns.boxplot(data=df_ood, x='strat_label', y='correct_cls_conf',
            order=strat_order, palette='Set2', ax=axes[0])
axes[0].set_title('Correct-class confidence\n(fully replaced — lower = more OoD)', fontweight='bold')
axes[0].set_xlabel('Strategy')
axes[0].set_ylabel('Confidence')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right')

# Max confidence
sns.boxplot(data=df_ood, x='strat_label', y='max_conf',
            order=strat_order, palette='Set2', ax=axes[1])
axes[1].set_title('Max confidence (fully replaced)\n(lower = more uniform/OoD)', fontweight='bold')
axes[1].set_xlabel('Strategy')
axes[1].set_ylabel('Max confidence')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')

# Prediction entropy
sns.boxplot(data=df_ood, x='strat_label', y='entropy',
            order=strat_order, palette='Set2', ax=axes[2])
axes[2].set_title('Prediction entropy (fully replaced)\n(higher = more uniform/uncertain)', fontweight='bold')
axes[2].set_xlabel('Strategy')
axes[2].set_ylabel('Entropy')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=20, ha='right')

plt.suptitle('Plot 4 — Out-of-Distribution Sensitivity per Replacement Strategy',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_ood_sensitivity.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 5 — Qualitative comparison grid
# Rows = replacement strategies, Columns = XAI methods
# Each cell: saliency overlay | masked image at 30% kept
# ─────────────────────────────────────────────────────────────────────────────
KEEP_FRAC = 0.30  # top-30% most salient pixels
qual_tensor, qual_pil, qual_cls, qual_name, _ = experiment_images[3]

print('Computing saliency for qualitative grid...')
qual_sals = {
    'GradCAM': compute_gradcam(model, qual_tensor, qual_cls),
    'Int. Grad.': compute_integrated_gradients(model, qual_tensor, qual_cls),
    'LIME': compute_lime(model, qual_pil, qual_cls),
    'SHAP': compute_shap(model, qual_tensor, BG_TENSOR[:20], qual_cls),
}

n_methods = len(qual_sals)
n_strats = len(STRAT_LABELS)
# 2 sub-columns per method (overlay + masked), plus original column
fig = plt.figure(figsize=(2 * n_methods * 2 + 2, n_strats * 2.2 + 1), dpi=DPI)
gs = gridspec.GridSpec(n_strats + 1, 2 * n_methods + 1,
                        hspace=0.05, wspace=0.05)

# Header: original
ax_orig = fig.add_subplot(gs[0, 0])
ax_orig.imshow(np.array(qual_pil.resize((IMG_SIZE, IMG_SIZE))))
ax_orig.set_title('Original', fontsize=8, fontweight='bold')
ax_orig.axis('off')

# Column headers
for col_i, mname in enumerate(qual_sals.keys()):
    ax_h = fig.add_subplot(gs[0, 1 + 2*col_i:1 + 2*col_i + 2])
    ax_h.text(0.5, 0.5, mname, ha='center', va='center', fontsize=9, fontweight='bold')
    ax_h.axis('off')

for row_i, (strat, slabel) in enumerate(STRAT_LABELS.items()):
    rfn = get_replace_fn(strat, bg_mean_tensor=BG_MEAN)
    # Row label
    ax_lbl = fig.add_subplot(gs[row_i + 1, 0])
    ax_lbl.text(0.5, 0.5, slabel, ha='center', va='center', fontsize=8, fontweight='bold',
                rotation=0)
    ax_lbl.axis('off')

    for col_i, (mname, sal) in enumerate(qual_sals.items()):
        # Build top-30% mask
        threshold = np.quantile(sal, 1 - KEEP_FRAC)
        mask = (sal >= threshold).astype(np.float32)

        # Replaced image
        replaced_t = rfn(qual_tensor, mask)
        conf_kept = _get_confidence(model, replaced_t, qual_cls)

        # Overlay saliency
        ax_sal = fig.add_subplot(gs[row_i + 1, 1 + 2*col_i])
        ax_sal.imshow(overlay_saliency(qual_pil, sal))
        ax_sal.axis('off')
        if row_i == 0:
            ax_sal.set_title('Saliency', fontsize=7)

        # Masked image
        ax_msk = fig.add_subplot(gs[row_i + 1, 2 + 2*col_i])
        ax_msk.imshow(tensor_to_numpy_rgb(replaced_t))
        ax_msk.set_title(f'conf={conf_kept:.2f}', fontsize=7, color='navy')
        ax_msk.axis('off')

plt.suptitle(f'Plot 5 — Qualitative Comparison Grid: {qual_name}\n'
             f'(top {int(KEEP_FRAC*100)}% salient pixels kept, rest replaced)',
             fontsize=11, fontweight='bold', y=1.01)
plt.savefig('fig_qualitative_grid.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 6 — Rank correlation matrix across strategies
# For each pair of strategies, Kendall's tau on insertion AUC per image×method
# ─────────────────────────────────────────────────────────────────────────────
from scipy.stats import kendalltau

# Pivot: index=(img_idx, xai), columns=strategy, values=ins_auc
pivot_rank = df_save.pivot_table(
    values='ins_auc',
    index=['img_idx', 'xai'],
    columns='strategy'
).dropna()

strat_cols = STRATEGIES
n_s = len(strat_cols)
tau_matrix  = np.zeros((n_s, n_s))
pval_matrix = np.zeros((n_s, n_s))

for i, s1 in enumerate(strat_cols):
    for j, s2 in enumerate(strat_cols):
        if i == j:
            tau_matrix[i, j] = 1.0
            pval_matrix[i, j] = 0.0
        else:
            tau, pval = kendalltau(pivot_rank[s1], pivot_rank[s2])
            tau_matrix[i, j] = tau
            pval_matrix[i, j] = pval

slabels_ordered = [STRAT_LABELS[s] for s in strat_cols]
df_tau = pd.DataFrame(tau_matrix, index=slabels_ordered, columns=slabels_ordered)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=DPI)

mask_upper = np.triu(np.ones_like(tau_matrix, dtype=bool), k=1)
sns.heatmap(df_tau, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
            ax=axes[0], mask=mask_upper, square=True)
axes[0].set_title('Kendall τ — Insertion AUC rank correlation\nbetween replacement strategies',
                   fontweight='bold')

# Same for deletion AUC
pivot_rank_del = df_save.pivot_table(
    values='del_auc', index=['img_idx', 'xai'], columns='strategy').dropna()
tau_del = np.zeros((n_s, n_s))
for i, s1 in enumerate(strat_cols):
    for j, s2 in enumerate(strat_cols):
        if i == j:
            tau_del[i, j] = 1.0
        else:
            tau_del[i, j], _ = kendalltau(pivot_rank_del[s1], pivot_rank_del[s2])

df_tau_del = pd.DataFrame(tau_del, index=slabels_ordered, columns=slabels_ordered)
sns.heatmap(df_tau_del, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
            ax=axes[1], mask=mask_upper, square=True)
axes[1].set_title('Kendall τ — Deletion AUC rank correlation\nbetween replacement strategies',
                   fontweight='bold')

plt.suptitle('Plot 6 — Strategy Agreement: Rank Correlation of AUC Scores',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rank_correlation.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 9: Summary Table (LaTeX) ──────────────────────────────────────────
agg = df_save.groupby(['xai', 'strategy']).agg(
    ins_mean=('ins_auc', 'mean'),
    ins_std=('ins_auc', 'std'),
    del_mean=('del_auc', 'mean'),
    del_std=('del_auc', 'std')
).reset_index()

agg['Insertion AUC'] = agg.apply(lambda r: f"{r.ins_mean:.3f} ± {r.ins_std:.3f}", axis=1)
agg['Deletion AUC']  = agg.apply(lambda r: f"{r.del_mean:.3f} ± {r.del_std:.3f}", axis=1)
agg['XAI Method'] = agg['xai'].map(XAI_LABELS)
agg['Strategy']   = agg['strategy'].map(STRAT_LABELS)

table = agg[['XAI Method', 'Strategy', 'Insertion AUC', 'Deletion AUC']]\
    .sort_values(['XAI Method', 'Strategy'])

print('=' * 70)
print('SUMMARY TABLE')
print('=' * 70)
print(table.to_string(index=False))
print()

latex_str = table.to_latex(
    index=False,
    caption='Mean $\\pm$ std of Insertion and Deletion AUC across 50 images, '
            'comparing XAI methods and pixel replacement strategies.',
    label='tab:auc_results',
    column_format='llcc',
    escape=False,
    bold_rows=False
)
print('LaTeX table:')
print(latex_str)

with open('table_auc_results.tex', 'w') as f:
    f.write(latex_str)
print('LaTeX table saved to table_auc_results.tex')

## Section 10: Discussion

### Why replacement strategy matters: the OoD problem

Insertion and Deletion metrics fundamentally rely on **perturbing images** by replacing or removing pixels. The core assumption is: *if a region is truly important, its absence should decrease the model's confidence.*

However, this assumption breaks down when the replacement creates **out-of-distribution (OoD) inputs** — images the model never encountered during training. When pixels are set to black (zero), the resulting image may look nothing like any natural image, and the model's confidence scores may reflect its response to an OoD input rather than the absence of a discriminative feature.

### Key findings from the experiments

1. **Black pixel replacement** is the simplest and fastest baseline, but tends to produce the most OoD inputs (Plot 4). It may artificially inflate deletion AUC because the model rapidly loses confidence due to unrealistic inputs rather than truly missing features.

2. **Gaussian blur** is a smooth, relatively natural perturbation that preserves low-frequency structure. It tends to produce more stable curves and lower OoD distortion than black replacement.

3. **Mean background (SHAP-style)** replaces masked pixels with the training-set pixel mean. This can be more in-distribution than black pixels, but loses spatial structure entirely.

4. **Random noise** is the most aggressive OoD strategy. High insertion/deletion AUC scores under noise replacement may be unreliable because the model is essentially evaluated on pure noise.

5. **CV2 Telea inpainting** provides spatially coherent replacements by filling masked regions with content from surrounding pixels. This keeps the replaced image closer to the natural image manifold, making the metric more faithful to the actual information-theoretic importance of features.

6. **Strategy agreement (Plot 6):** High Kendall τ between two strategies means they agree on which XAI method is better. Low or negative τ means strategy choice can **reverse the ranking** of methods — a critical finding for metric reliability.

### Limitations of generative inpainting

- **Semantic leakage:** Inpainting models (especially SD-based) can hallucinate content that is semantically consistent with the class, inadvertently inserting discriminative features and biasing the metric.
- **Computational cost:** SD inpainting is 100-1000× slower than black replacement, making large-scale experiments expensive.
- **Stochasticity:** Generative models are non-deterministic; the same mask applied twice produces different results, increasing variance.
- **Checkerboard artifacts:** Telea and other local inpainters can produce visible artifacts at mask boundaries, especially for large masks.

### Recommendations for the thesis

1. **Use CV2 Telea as the primary replacement strategy** for the main results — it provides spatially coherent replacements with minimal computational overhead.
2. **Report results for all strategies** to demonstrate the sensitivity of the metrics.
3. **Report OoD sensitivity plots** as a diagnostic tool to justify the choice of strategy.
4. **Avoid random noise** as a primary strategy — it is too aggressive and unpredictable.
5. **Consider including Stable Diffusion inpainting** as an aspirational comparison to show the direction of future work.
6. **For rank-based comparisons**, use Kendall τ to show that strategy choice affects method rankings — this is the core research contribution.

In [ ]:
# ── Appendix: Optional Stable Diffusion Inpainting Setup ──────────────────────
# Run this cell only if you have sufficient VRAM (>10GB) on Colab.
# The SD2-inpainting model is ~5GB.

USE_SD_INPAINTING = False  # Set to True to enable

if USE_SD_INPAINTING:
    from diffusers import StableDiffusionInpaintPipeline
    import torch

    SD_PIPE = StableDiffusionInpaintPipeline.from_pretrained(
        'stabilityai/stable-diffusion-2-inpainting',
        torch_dtype=torch.float16
    ).to(DEVICE)
    SD_PIPE.enable_attention_slicing()  # reduce VRAM usage

    # Test on one image
    test_t, test_pil, test_cls, test_name, _ = experiment_images[0]
    test_mask = demo_mask
    sd_result = replace_inpaint_sd(test_t, test_mask, SD_PIPE)

    fig, axes = plt.subplots(1, 3, figsize=(10, 4), dpi=DPI)
    axes[0].imshow(tensor_to_numpy_rgb(test_t))
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow((1 - test_mask), cmap='gray')
    axes[1].set_title('Mask (white=replaced)')
    axes[1].axis('off')

    axes[2].imshow(tensor_to_numpy_rgb(sd_result))
    axes[2].set_title('SD Inpainting Result')
    axes[2].axis('off')

    plt.suptitle('Stable Diffusion 2 Inpainting', fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig_sd_inpainting.png', dpi=DPI, bbox_inches='tight')
    plt.show()

    # Re-run full experiment including SD strategy
    # (Add 'inpaint_sd' to STRATEGIES and re-run Section 7)
    print('SD inpainting ready. Add inpaint_sd to STRATEGIES and re-run Section 7.')
else:
    print('SD inpainting disabled. Set USE_SD_INPAINTING=True to enable.')
    print('Requires ~10GB VRAM. Use Colab A100 runtime if available.')